# Verify 5x5 Connectivity Coordinate Mapping

This notebook validates that `connectivity_matrix_5x5` fills all 24 center-excluding channels with the intended `(dr, dc)` offsets.

In [ ]:
import torch
import autorootcwd  # Auto-changes cwd to repo root on import

from connect_loss import connectivity_matrix_5x5

device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# Build a deterministic mask with sequential values in range [0, 1)
H, W = 10, 10
# Generate 0.0000 to 0.9999 (100 values evenly spaced)
mask2d = torch.linspace(0, 1 - 1/(H*W), H*W, dtype=torch.float32, device=device).view(H, W)

multimask = mask2d.unsqueeze(0).unsqueeze(0)  # (B=1,C=1,H,W)
conn = connectivity_matrix_5x5(multimask, class_num=1)
print('sample mask (first 3 rows):')
print(mask2d[:3].cpu().numpy())
print('...')
print(f'multimask shape: {tuple(multimask.shape)}')
print(f'conn shape: {tuple(conn.shape)}')
print(f'Value range: [{mask2d.min():.4f}, {mask2d.max():.4f}]')

In [ ]:
# Independent expected-value check:
# first 8 channels follow canonical 8-connectivity order, remaining channels cover the
# rest of the 5x5 neighborhood excluding the center.
B, C, H, W = multimask.shape
assert B == 1 and C == 1
mask = multimask[0, 0]

canonical_offsets = [
    (1, 1),
    (1, 0),
    (1, -1),
    (0, 1),
    (0, -1),
    (-1, 1),
    (-1, 0),
    (-1, -1),
]
offsets = list(canonical_offsets)
for dr in range(-2, 3):
    for dc in range(-2, 3):
        if dr == 0 and dc == 0:
            continue
        if (dr, dc) not in offsets:
            offsets.append((dr, dc))

assert conn.shape[1] == 24

mismatches = []
for ch, (dr, dc) in enumerate(offsets):
    expected = torch.zeros((H, W), dtype=mask.dtype, device=mask.device)
    for r in range(H):
        for c in range(W):
            rr = r - dr
            cc = c - dc
            if 0 <= rr < H and 0 <= cc < W:
                expected[r, c] = mask[r, c] * mask[rr, cc]

    got = conn[0, ch]
    if not torch.equal(got, expected):
        diff_count = int((got != expected).sum().item())
        mismatches.append((ch, dr, dc, diff_count))

if mismatches:
    print('FAIL: mismatches found')
    for m in mismatches[:10]:
        print('channel=%d dr=%d dc=%d diff_pixels=%d' % m)
else:
    print('PASS: all 24 channels match expected coordinate mapping')

In [ ]:
# Print channel-to-offset table for reference
for ch, (dr, dc) in enumerate(offsets):
    nz = int((conn[0, ch] > 0).sum().item())
    print(f'ch={ch:02d} -> (dr={dr:+d}, dc={dc:+d}), nonzero={nz}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualize all 24 channels in a 5x5 grid with pixel values displayed
fig, axes = plt.subplots(5, 5, figsize=(18, 18))
fig.suptitle('24 Directional Channels - 5x5 Offset Grid (10x10 size with Pixel Values)',
             fontsize=14, fontweight='bold')

for idx, (dr, dc) in enumerate(offsets):
    row = idx // 5
    col = idx % 5
    ax = axes[row, col]

    # Display channel
    ch_data = conn[0, idx].cpu().numpy()
    im = ax.imshow(ch_data, cmap='viridis', interpolation='nearest')
    ax.set_title(f'ch={idx} (dr={dr:+d}, dc={dc:+d})', fontsize=10, fontweight='bold')

    # Add pixel value text inside each cell (with 4 decimal places)
    for i in range(ch_data.shape[0]):
        for j in range(ch_data.shape[1]):
            val = ch_data[i, j]
            if val > 1e-6:  # Only show non-zero values for clarity
                # Choose text color based on background brightness
                color = 'white' if val < ch_data.max() / 2 else 'black'
                ax.text(j, i, f'{val:.4f}', ha='center', va='center',
                        color=color, fontsize=6, fontweight='bold')

    ax.set_xticks([])
    ax.set_yticks([])

    # Add colorbar
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

print('Visualization complete: 24 channels (10x10) with 4-digit float pixel values displayed')

## Why Do Pixel Values Change?

### Core Formula
각 채널의 픽셀 값은 **두 마스크의 곱셈**으로 계산됩니다:

```
conn[ch, r, c] = mask[r, c] × mask[r-dr, c-dc]
```

여기서:
- `mask[r, c]`: 위치 (r, c)의 원본 값
- `mask[r-dr, c-dc]`: 오프셋 (dr, dc)만큼 이동된 위치의 값

### 구체적 예시
원본 마스크가 0.0000~0.9999 범위의 실수이므로, 위치별로:
- mask[5, 5] ≈ 0.5555
- mask[3, 3] ≈ 0.3333
- mask[6, 6] ≈ 0.6666

**Channel 0 (dr=+2, dc=+2):**
- conn[0, 5, 5] = mask[5, 5] × mask[3, 3] ≈ 0.5555 × 0.3333 ≈ **0.185165**

**Channel 12 (dr=0, dc=0) 중앙:**
- conn[12, 5, 5] = mask[5, 5] × mask[5, 5] ≈ 0.5555 × 0.5555 ≈ **0.308610** (제곱)

**Channel 17 (dr=-1, dc=0) 위쪽:**
- conn[17, 5, 5] = mask[5, 5] × mask[6, 5] ≈ 0.5555 × 0.6666 ≈ **0.370340**

따라서 **원본 값 0.5555**가 채널마다 다른 값과 곱해지므로
각 채널마다 **완전히 다른 픽셀 값**이 생성됩니다!

In [ ]:
# Verify pixel value multiplication examples with floating-point values
test_pos = (5, 5)  # Example position (r=5, c=5)
mask_val = float(mask[test_pos].item())

print(f'=== Pixel Value Multiplication at position {test_pos} ===')
print(f'Original mask value: mask[{test_pos}] = {mask_val:.4f}\n')

# Select a few example channels to demonstrate
example_channels = [0, 6, 12, 17, 24]  # corner, edge, center, edge, corner

for ch in example_channels:
    dr, dc = offsets[ch]
    src_r, src_c = test_pos[0] - dr, test_pos[1] - dc

    # Check if source position is in bounds
    if 0 <= src_r < H and 0 <= src_c < W:
        src_val = float(mask[src_r, src_c].item())
        result = float(conn[0, ch, test_pos[0], test_pos[1]].item())
        print(f'Channel {ch:2d} (dr={dr:+d}, dc={dc:+d}):')
        print(f'  conn[{ch}, {test_pos}] = mask[{test_pos}] × mask[{src_r},{src_c}]')
        print(f'                    = {mask_val:.4f} × {src_val:.4f} = {result:.6f}')
    else:
        print(f'Channel {ch:2d} (dr={dr:+d}, dc={dc:+d}): out of bounds')
    print()